# 面试题：Learning to Rank 中 LambdaRank 怎样从指标推到梯度？

本 Notebook 用 PyTorch 基础层手写 query group、DCG/nDCG、按当前排序位置计算 `ΔnDCG`、加权 pairwise logistic loss、训练、稳定排序和发布 wrapper。不调用 LightGBM/XGBoost/TorchRank。

合成特征只验证 Lambda 权重、梯度和 group split；真实点击排序还要处理位置偏差、延迟反馈、多目标、校准和在线探索。

In [ ]:
import copy,hashlib,io,json,math,random,warnings  # 导入本单元所需的依赖。
from dataclasses import dataclass  # 导入本单元所需的依赖。
from types import MappingProxyType  # 导入本单元所需的依赖。
warnings.filterwarnings("ignore",message="The pynvml package is deprecated")  # 计算并保存当前步骤的中间状态。
import numpy as np  # 导入本单元所需的依赖。
import torch  # 导入本单元所需的依赖。
from torch import nn  # 导入本单元所需的依赖。
import torch.nn.functional as F  # 导入本单元所需的依赖。
SEED72=7201  # 计算并保存当前步骤的中间状态。
random.seed(SEED72); np.random.seed(SEED72); torch.manual_seed(SEED72); torch.set_num_threads(1)  # 执行当前语句以推进本节示例。
def canonical72(x): return json.dumps(x,sort_keys=True,separators=(",",":"))  # 定义本节可复用的核心函数。
def sha72(x): return hashlib.sha256(x).hexdigest()  # 定义本节可复用的核心函数。
assert torch.get_num_threads()==1  # 用受控断言验证关键不变量。

## 1. Query group 与数据切分

排序样本单位是 query，不是独立 doc。每个 query 有 7 个候选、4 维特征、0–3 relevance 和唯一 doc ID。train/validation/test 按 query family 切分，禁止同一 query 的候选跨 split，否则 pair 和 nDCG 都泄漏。

特征来自带噪潜在质量、query-doc match、freshness 和长度偏差，没有把 grade 直接作为输入。

In [ ]:
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class QueryGroup72: query_id:str; family:str; doc_ids:tuple; features:torch.Tensor; grades:torch.Tensor  # 定义承载本节状态与行为的数据结构。
def make_group72(index,seed):  # 定义本节可复用的核心函数。
    g=torch.Generator().manual_seed(seed); x=torch.randn(7,4,generator=g)  # 计算并保存当前步骤的中间状态。
    latent=1.5*x[:,0]+.9*x[:,1]-.45*x[:,2]+.3*x[:,3]+.15*torch.randn(7,generator=g)  # 计算并保存当前步骤的中间状态。
    order=torch.argsort(latent,descending=True); grades=torch.zeros(7,dtype=torch.long); grades[order[0]]=3; grades[order[1:3]]=2; grades[order[3:5]]=1  # 计算并保存当前步骤的中间状态。
    return QueryGroup72(f"q{index}",f"family-{index}",tuple(f"q{index}-d{j}" for j in range(7)),x,grades)  # 返回当前分支计算出的结果。
train_groups72=[make_group72(i,7300+i) for i in range(18)]; val_groups72=[make_group72(i,7300+i) for i in range(18,22)]; test_groups72=[make_group72(i,7300+i) for i in range(22,28)]  # 计算并保存当前步骤的中间状态。
assert all(g.features.shape==(7,4) and g.grades.shape==(7,) for g in train_groups72+val_groups72+test_groups72)  # 用受控断言验证关键不变量。
assert set(g.family for g in train_groups72).isdisjoint(g.family for g in val_groups72+test_groups72)  # 用受控断言验证关键不变量。
assert set(d for g in train_groups72 for d in g.doc_ids).isdisjoint(d for g in test_groups72 for d in g.doc_ids)  # 用受控断言验证关键不变量。
assert all(torch.bincount(g.grades,minlength=4).tolist()==[2,2,2,1] for g in train_groups72)  # 用受控断言验证关键不变量。

## 2. 手写 DCG、IDCG 与 nDCG

`DCG@k = Σ(2^rel-1)/log2(rank+1)`，nDCG 用同 query 的理想排序归一化。空 gold/IDCG=0 的策略必须明确，这里返回 0。排序 tie 用 doc ID 保证副本一致。

指标按 query 计算再宏平均，不能把大候选 query 的所有 pair 混在一起做微平均。

In [ ]:
def dcg72(grades,k=None):  # 定义本节可复用的核心函数。
    g=torch.as_tensor(grades,dtype=torch.float32); g=g if k is None else g[:k]  # 计算并保存当前步骤的中间状态。
    if g.ndim!=1: raise ValueError("grade_shape")  # 按当前条件选择后续控制路径。
    return ((2**g-1)/torch.log2(torch.arange(len(g),dtype=torch.float32)+2)).sum()  # 返回当前分支计算出的结果。
def ndcg72(scores,grades,k=None):  # 定义本节可复用的核心函数。
    scores=torch.as_tensor(scores); grades=torch.as_tensor(grades)  # 计算并保存当前步骤的中间状态。
    if scores.shape!=grades.shape or scores.ndim!=1: raise ValueError("metric_shape")  # 按当前条件选择后续控制路径。
    order=sorted(range(len(scores)),key=lambda i:(-float(scores[i]),i)); actual=dcg72(grades[order],k); ideal=dcg72(torch.sort(grades,descending=True).values,k)  # 计算并保存当前步骤的中间状态。
    return float(actual/ideal) if ideal>0 else 0.  # 返回当前分支计算出的结果。
assert math.isclose(ndcg72(torch.tensor([3.,2.,1.]),torch.tensor([3,2,1])),1.)  # 用受控断言验证关键不变量。
assert ndcg72(torch.tensor([1.,2.,3.]),torch.tensor([3,2,1]))<1.  # 用受控断言验证关键不变量。
assert ndcg72(torch.zeros(3),torch.zeros(3))==0. and dcg72([0,0]).item()==0.  # 用受控断言验证关键不变量。
try: ndcg72(torch.zeros(2),torch.zeros(3)); raise AssertionError("metric mismatch accepted")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="metric_shape"  # 捕获预期异常并验证失败分支。

## 3. 小型打分网络与逐文档前向

Ranker 对每个候选独立输出标量，query 内再排序。网络本身不看 grade 或其他候选；listwise 交互来自 loss。shape 合同为 `[N,4] -> [N]`，拒绝 NaN 和错误维度。

生产中可能加入 query encoder、cross features 或 Transformer，但必须保证 serving 特征与训练 Point-in-Time 语义一致。

In [ ]:
class Ranker72(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self,input_dim=4,hidden=12): super().__init__(); self.input_dim=input_dim; self.net=nn.Sequential(nn.Linear(input_dim,hidden),nn.Tanh(),nn.Linear(hidden,1))  # 定义本节可复用的核心函数。
    def forward(self,x):  # 定义本节可复用的核心函数。
        if x.ndim!=2 or x.shape[1]!=self.input_dim or not torch.isfinite(x).all(): raise ValueError("ranker_input_contract")  # 按当前条件选择后续控制路径。
        return self.net(x).squeeze(-1)  # 返回当前分支计算出的结果。
ranker_probe72=Ranker72(); score_probe72=ranker_probe72(train_groups72[0].features)  # 计算并保存当前步骤的中间状态。
assert score_probe72.shape==(7,) and torch.isfinite(score_probe72).all()  # 用受控断言验证关键不变量。
score_probe72.sum().backward(); assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in ranker_probe72.parameters())  # 执行当前语句以推进本节示例。
try: ranker_probe72(torch.zeros(2,3)); raise AssertionError("wrong feature dim accepted")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="ranker_input_contract"  # 捕获预期异常并验证失败分支。

## 4. `ΔnDCG`：交换一对文档会损失多少指标

LambdaRank 关注不同 relevance pair，并按交换它们在当前 rank 位置造成的 `|ΔnDCG|` 加权。高位交换和 grade 差大的 pair 权重更高。位置来自当前 score 排序，权重在每次 forward 重算但不反向传播。

这是把非光滑排序指标转为可优化 pairwise surrogate 的关键；它不是 nDCG 的真实解析梯度。

In [ ]:
def delta_ndcg72(scores,grades):  # 定义本节可复用的核心函数。
    n=len(scores); order=torch.argsort(scores,descending=True,stable=True); positions=torch.empty(n,dtype=torch.long); positions[order]=torch.arange(n)  # 计算并保存当前步骤的中间状态。
    gains=2**grades.float()-1; discounts=1/torch.log2(torch.arange(n,dtype=torch.float32)+2); idcg=dcg72(torch.sort(grades,descending=True).values)  # 计算并保存当前步骤的中间状态。
    delta=torch.zeros(n,n)  # 计算并保存当前步骤的中间状态。
    if idcg>0:  # 按当前条件选择后续控制路径。
        for i in range(n):  # 遍历输入元素以累积或检查结果。
            for j in range(n): delta[i,j]=abs((gains[i]-gains[j])*(discounts[positions[i]]-discounts[positions[j]])/idcg)  # 遍历输入元素以累积或检查结果。
    return delta  # 返回当前分支计算出的结果。
scores_oracle72=torch.tensor([3.,2.,1.]); grades_oracle72=torch.tensor([3,0,2]); delta_oracle72=delta_ndcg72(scores_oracle72,grades_oracle72)  # 计算并保存当前步骤的中间状态。
assert delta_oracle72.shape==(3,3) and torch.allclose(delta_oracle72,delta_oracle72.T)  # 用受控断言验证关键不变量。
assert torch.allclose(torch.diag(delta_oracle72),torch.zeros(3)) and delta_oracle72[0,1]>delta_oracle72[1,2]  # 用受控断言验证关键不变量。
assert torch.isfinite(delta_oracle72).all() and bool((delta_oracle72>=0).all())  # 用受控断言验证关键不变量。

## 5. 加权 pairwise logistic loss 与梯度方向

对每个 `grade_i > grade_j`，最小化 `ΔnDCG_ij * softplus(-(s_i-s_j))`。只枚举一次有序 pair，按权重和归一化，避免 query 候选数改变 loss 尺度。

手工 oracle 验证：提高相关文档 score 会降低 loss；相同 grade 不产生 pair；全零 grade 返回与 score 图相连的零 loss，训练循环无需特殊分支。

In [ ]:
def lambda_loss72(scores,grades):  # 定义本节可复用的核心函数。
    if scores.shape!=grades.shape or scores.ndim!=1: raise ValueError("lambda_shape")  # 按当前条件选择后续控制路径。
    delta=delta_ndcg72(scores.detach(),grades); terms=[]  # 计算并保存当前步骤的中间状态。
    for i in range(len(scores)):  # 遍历输入元素以累积或检查结果。
        for j in range(len(scores)):  # 遍历输入元素以累积或检查结果。
            if grades[i]>grades[j] and delta[i,j]>0: terms.append(delta[i,j]*F.softplus(-(scores[i]-scores[j])))  # 按当前条件选择后续控制路径。
    return torch.stack(terms).sum()/(torch.stack([delta[i,j] for i in range(len(scores)) for j in range(len(scores)) if grades[i]>grades[j] and delta[i,j]>0]).sum()+1e-8) if terms else scores.sum()*0  # 返回当前分支计算出的结果。
low_loss72=lambda_loss72(torch.tensor([3.,0.,1.]),torch.tensor([3,0,2])); high_loss72=lambda_loss72(torch.tensor([0.,3.,1.]),torch.tensor([3,0,2]))  # 计算并保存当前步骤的中间状态。
assert low_loss72<high_loss72 and lambda_loss72(torch.zeros(3),torch.zeros(3,dtype=torch.long))==0  # 用受控断言验证关键不变量。
grad_scores72=torch.tensor([0.,0.,0.],requires_grad=True); lambda_loss72(grad_scores72,torch.tensor([3,1,0])).backward()  # 计算并保存当前步骤的中间状态。
assert grad_scores72.grad[0]<0 and grad_scores72.grad[-1]>0 and abs(float(grad_scores72.grad.sum()))<1e-6  # 用受控断言验证关键不变量。

## 6. 按 query 训练与 validation 选择

每步对 train query 的 loss 宏平均；validation 只评估 nDCG，不参与梯度。保存初始/最终 test 指标，证明受控数据上的排序改善。真实训练还需要 early stopping、query weighting 和多随机种子。

不能把 doc 行随机打散再切分；optimizer 的 batch 应是 query group 或正确 padding/mask 的多 query batch。

In [ ]:
def mean_ndcg72(model,groups):  # 定义本节可复用的核心函数。
    with torch.no_grad(): return float(np.mean([ndcg72(model(g.features),g.grades) for g in groups]))  # 在受管理的上下文中执行操作。
torch.manual_seed(SEED72); model72=Ranker72(); optimizer72=torch.optim.Adam(model72.parameters(),lr=.025)  # 计算并保存当前步骤的中间状态。
initial_test72=mean_ndcg72(model72,test_groups72); val_history72=[]  # 计算并保存当前步骤的中间状态。
for step72 in range(60):  # 遍历输入元素以累积或检查结果。
    losses=[lambda_loss72(model72(g.features),g.grades) for g in train_groups72]; loss72=torch.stack(losses).mean()  # 计算并保存当前步骤的中间状态。
    optimizer72.zero_grad(set_to_none=True); loss72.backward(); torch.nn.utils.clip_grad_norm_(model72.parameters(),5.); optimizer72.step()  # 计算并保存当前步骤的中间状态。
    if step72%10==0: val_history72.append(mean_ndcg72(model72,val_groups72))  # 按当前条件选择后续控制路径。
final_test72=mean_ndcg72(model72,test_groups72)  # 计算并保存当前步骤的中间状态。
assert final_test72>initial_test72+.05 and final_test72>.95  # 用受控断言验证关键不变量。
assert val_history72[-1]>val_history72[0] and all(math.isfinite(v) for v in val_history72)  # 用受控断言验证关键不变量。
assert any(p.grad is not None and torch.isfinite(p.grad).all() for p in model72.parameters())  # 用受控断言验证关键不变量。

## 7. 稳定排序服务与干预测试

Published ranker 接收命名 doc IDs 和 `[N,4]` 特征，限制候选数、检查唯一 ID/finite，并以 `(-score, doc_id)` 排序。改变一条候选特征不能改变其他候选的原始 score，但可能改变它们的 rank，这是排序的正常竞争效应。

tie-break 必须明确；过滤/ACL 应在返回前执行，且 trace 要保留候选集版本和特征时间戳。

In [ ]:
class PublishedRanker72:  # 定义承载本节状态与行为的数据结构。
    def __init__(self,model,max_candidates=50): self._model=model.eval(); self.max_candidates=max_candidates  # 定义本节可复用的核心函数。
    @torch.no_grad()  # 为下方定义附加声明式配置。
    def rank(self,doc_ids,features):  # 定义本节可复用的核心函数。
        x=torch.as_tensor(features,dtype=torch.float32)  # 计算并保存当前步骤的中间状态。
        if not 1<=len(doc_ids)<=self.max_candidates or len(set(doc_ids))!=len(doc_ids) or x.shape!=(len(doc_ids),4) or not torch.isfinite(x).all(): raise ValueError("ranking_request_contract")  # 按当前条件选择后续控制路径。
        scores=self._model(x); order=sorted(range(len(doc_ids)),key=lambda i:(-float(scores[i]),doc_ids[i]))  # 计算并保存当前步骤的中间状态。
        return tuple((doc_ids[i],float(scores[i])) for i in order)  # 返回当前分支计算出的结果。
pub_probe72=PublishedRanker72(model72); group_probe72=test_groups72[0]; served72=pub_probe72.rank(group_probe72.doc_ids,group_probe72.features)  # 计算并保存当前步骤的中间状态。
assert len(served72)==7 and len({d for d,_ in served72})==7  # 用受控断言验证关键不变量。
changed_features72=group_probe72.features.clone(); changed_features72[0]+=1; old_scores72=dict(served72); new_scores72=dict(pub_probe72.rank(group_probe72.doc_ids,changed_features72))  # 计算并保存当前步骤的中间状态。
assert all(math.isclose(old_scores72[d],new_scores72[d],abs_tol=1e-6) for d in group_probe72.doc_ids[1:])  # 用受控断言验证关键不变量。
try: pub_probe72.rank(("a","a"),torch.zeros(2,4)); raise AssertionError("duplicate IDs accepted")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="ranking_request_contract"  # 捕获预期异常并验证失败分支。

## 8. 模型制品、特征语义与回归集

state 摘要逐 tensor 绑定 key/dtype/shape/bytes；manifest 绑定 feature 顺序、变换、query split、grade 语义、loss、seed 和指标。只存 checkpoint 不存特征合同，会产生训练/服务错位。

包外 registry 模拟发布签名根。loader 从实际 state 重算 release digest，拒绝攻击者替换权重后重签包内字段。

In [ ]:
def state_digest72(state):  # 定义本节可复用的核心函数。
    h=hashlib.sha256()  # 计算并保存当前步骤的中间状态。
    for k,v in sorted(state.items()):  # 遍历输入元素以累积或检查结果。
        a=v.detach().cpu().contiguous().numpy(); h.update(k.encode()); h.update(str(a.dtype).encode()); h.update(canonical72(list(a.shape)).encode()); h.update(a.tobytes())  # 计算并保存当前步骤的中间状态。
    return h.hexdigest()  # 返回当前分支计算出的结果。
state72={k:v.detach().clone() for k,v in model72.state_dict().items()}  # 计算并保存当前步骤的中间状态。
manifest72={"artifact_id":"lambdarank-demo-v1","config":{"input_dim":4,"hidden":12},"features":["latent_match_1","latent_match_2","freshness_cost","length_signal"],"grade":[0,1,2,3],"loss":"delta_ndcg_weighted_pairwise_softplus","training":{"steps":60,"optimizer":"Adam","lr":.025,"grad_clip":5.},"train_queries":[g.query_id for g in train_groups72],"validation_queries":[g.query_id for g in val_groups72],"seed":SEED72}  # 计算并保存当前步骤的中间状态。
release72=sha72(canonical72({"manifest":manifest72,"state":state_digest72(state72)}).encode()); TRUST72=MappingProxyType({manifest72["artifact_id"]:release72})  # 计算并保存当前步骤的中间状态。
def load_ranker72(m,state):  # 定义本节可复用的核心函数。
    actual=sha72(canonical72({"manifest":m,"state":state_digest72(state)}).encode())  # 计算并保存当前步骤的中间状态。
    if TRUST72.get(m.get("artifact_id"))!=actual: raise RuntimeError("untrusted_ranker")  # 按当前条件选择后续控制路径。
    model=Ranker72(**m["config"]); model.load_state_dict(state); return PublishedRanker72(model)  # 计算并保存当前步骤的中间状态。
published72=load_ranker72(manifest72,state72)  # 计算并保存当前步骤的中间状态。
assert published72.rank(group_probe72.doc_ids,group_probe72.features)==served72 and isinstance(TRUST72,MappingProxyType)  # 用受控断言验证关键不变量。
forged_state72={k:v.clone() for k,v in state72.items()}; first72=next(iter(forged_state72)); forged_state72[first72].view(-1)[0]+=1  # 计算并保存当前步骤的中间状态。
try: load_ranker72(manifest72,forged_state72); raise AssertionError("forged ranker accepted")  # 尝试执行可能失败的受控操作。
except RuntimeError as e: assert str(e)=="untrusted_ranker"  # 捕获预期异常并验证失败分支。
print({"initial_test_ndcg":round(initial_test72,4),"final_test_ndcg":round(final_test72,4),"val_last":round(val_history72[-1],4)})  # 执行当前语句以推进本节示例。

## 9. 复杂度、失败模式与来源

朴素 pair 枚举是 `O(n²)`，大候选 query 要采 pair 或使用高效 Lambda 实现。常见错误：doc 级随机切分、把 grade 当特征、IDCG 跨 query、忽略 ties、同 grade 建 pair、点击偏差当 relevance、候选集变化却仍比较 nDCG。

- Burges et al., [Learning to Rank using Gradient Descent](https://icml.cc/2015/wp-content/uploads/2015/06/icml_ranking.pdf)，RankNet/LambdaRank 背景。
- Burges, [From RankNet to LambdaRank to LambdaMART](https://www.microsoft.com/en-us/research/wp-content/uploads/2016/02/MSR-TR-2010-82.pdf)。
- Järvelin & Kekäläinen, [Cumulated gain-based evaluation of IR techniques](https://dl.acm.org/doi/10.1145/582415.582418)。